In [ ]:
# Imports
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from PIL import Image

In [ ]:
# GradCAM imports
# If this cell fails, run: pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
# Device setup
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
# Reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [ ]:
# Create results folders
os.makedirs("results", exist_ok=True)
os.makedirs("results/confusion_matrix", exist_ok=True)
os.makedirs("results/gradcam", exist_ok=True)
os.makedirs("results/sample_predictions", exist_ok=True)

print("Results folders ready")


In [ ]:
# Load dataset and rebuild the same split used earlier
# This notebook is self contained, so it does not depend on running 01_data_prep first

dataDir = "data"

valTransform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

fullDataset = datasets.ImageFolder(root=dataDir, transform=valTransform)

datasetSize = len(fullDataset)
trainSize = int(0.7 * datasetSize)
valSize = int(0.15 * datasetSize)
testSize = datasetSize - trainSize - valSize

trainDataset, valDataset, testDataset = random_split(
    fullDataset,
    [trainSize, valSize, testSize],
    generator=torch.Generator().manual_seed(42)
)

batchSize = 16
testLoader = DataLoader(testDataset, batch_size=batchSize, shuffle=False, num_workers=0)

classNames = fullDataset.classes
print("Class names:", classNames)
print("Test size:", len(testDataset))

In [ ]:
# Load model architecture
model = models.densenet121(pretrained=False)
numFeatures = model.classifier.in_features
model.classifier = nn.Linear(numFeatures, 4)
model = model.to(device)

print("Model created")

In [ ]:
# Load best checkpoint
checkpointPath = "models/densenetAllBest.pth"

if not os.path.exists(checkpointPath):
    raise FileNotFoundError(f"Best checkpoint not found at: {checkpointPath}")

checkpoint = torch.load(checkpointPath, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded best model from:", checkpointPath)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Checkpoint val loss:", checkpoint.get("val_loss", "unknown"))

In [ ]:
# Helper function to unnormalize image tensors for display
def unnormalizeImage(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.cpu().numpy().transpose(1, 2, 0)
    img = (img * std) + mean
    img = np.clip(img, 0, 1)
    return img

In [ ]:
# Evaluate on the test set
allPreds = []
allLabels = []

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in testLoader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        allPreds.extend(preds.cpu().numpy())
        allLabels.extend(labels.cpu().numpy())

testAccuracy = correct / total
print(f"Test accuracy: {testAccuracy:.4f}")

In [ ]:
# Classification report
report = classification_report(allLabels, allPreds, target_names=classNames, digits=4)
print(report)

with open("results/classification_report.txt", "w") as f:
    f.write(report)

print("Saved classification report to results/classification_report.txt")

In [ ]:
# Confusion matrix
cm = confusion_matrix(allLabels, allPreds)

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classNames)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("results/confusion_matrix/confusion_matrix.png", dpi=300)
plt.show()

print("Saved confusion matrix to results/confusion_matrix/confusion_matrix.png")

In [ ]:
# Show a few test predictions
def showSamplePredictions(loader, model, classes, numImages=6):
    model.eval()
    shown = 0

    plt.figure(figsize=(15, 6))

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for i in range(inputs.size(0)):
                if shown >= numImages:
                    plt.tight_layout()
                    plt.savefig("results/sample_predictions/sample_predictions.png", dpi=300)
                    plt.show()
                    print("Saved sample predictions to results/sample_predictions/sample_predictions.png")
                    return

                img = unnormalizeImage(inputs[i])
                plt.subplot(2, 3, shown + 1)
                plt.imshow(img)
                plt.title(f"True: {classes[labels[i].item()]}\nPred: {classes[preds[i].item()]}")
                plt.axis("off")
                shown += 1

showSamplePredictions(testLoader, model, classNames, numImages=6)

In [ ]:
# GradCAM setup
# For DenseNet121, the final convolutional block is model.features
targetLayers = [model.features]

cam = GradCAM(model=model, target_layers=targetLayers)

print("GradCAM ready")

In [ ]:
# Generate GradCAM for a few correctly classified examples
def saveGradCamExamples(loader, model, cam, classes, maxPerClass=2):
    model.eval()

    savedPerClass = {className: 0 for className in classes}

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        for i in range(inputs.size(0)):
            trueLabel = labels[i].item()
            predLabel = preds[i].item()

            # only save correctly classified examples
            if trueLabel != predLabel:
                continue

            className = classes[trueLabel]
            if savedPerClass[className] >= maxPerClass:
                continue

            rgbImg = unnormalizeImage(inputs[i])

            targets = [ClassifierOutputTarget(predLabel)]
            grayscaleCam = cam(input_tensor=inputs[i].unsqueeze(0), targets=targets)[0]

            visualization = show_cam_on_image(rgbImg.astype(np.float32), grayscaleCam, use_rgb=True)

            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(rgbImg)
            axes[0].set_title(f"Original\n{className}")
            axes[0].axis("off")

            axes[1].imshow(visualization)
            axes[1].set_title(f"GradCAM\nPred: {classes[predLabel]}")
            axes[1].axis("off")

            plt.tight_layout()

            savePath = f"results/gradcam/{className}_{savedPerClass[className] + 1}.png"
            plt.savefig(savePath, dpi=300)
            plt.show()
            plt.close(fig)

            savedPerClass[className] += 1

        if all(count >= maxPerClass for count in savedPerClass.values()):
            break

    print("Saved GradCAM examples:")
    for className, count in savedPerClass.items():
        print(className, count)

saveGradCamExamples(testLoader, model, cam, classNames, maxPerClass=2)


In [ ]:
# Optional: Generate GradCAM for misclassified examples too
def saveMisclassifiedGradCamExamples(loader, model, cam, classes, maxImages=4):
    model.eval()
    saved = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        for i in range(inputs.size(0)):
            trueLabel = labels[i].item()
            predLabel = preds[i].item()

            if trueLabel == predLabel:
                continue

            rgbImg = unnormalizeImage(inputs[i])

            targets = [ClassifierOutputTarget(predLabel)]
            grayscaleCam = cam(input_tensor=inputs[i].unsqueeze(0), targets=targets)[0]
            visualization = show_cam_on_image(rgbImg.astype(np.float32), grayscaleCam, use_rgb=True)

            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(rgbImg)
            axes[0].set_title(f"Original\nTrue: {classes[trueLabel]}")
            axes[0].axis("off")

            axes[1].imshow(visualization)
            axes[1].set_title(f"GradCAM\nPred: {classes[predLabel]}")
            axes[1].axis("off")

            plt.tight_layout()

            savePath = f"results/gradcam/misclassified_{saved + 1}.png"
            plt.savefig(savePath, dpi=300)
            plt.show()
            plt.close(fig)

            saved += 1

            if saved >= maxImages:
                print(f"Saved {saved} misclassified GradCAM examples")
                return

saveMisclassifiedGradCamExamples(testLoader, model, cam, classNames, maxImages=4)

In [ ]:
# Save a short summary file
summaryText = []
summaryText.append(f"Test accuracy: {testAccuracy:.4f}")
summaryText.append("")
summaryText.append("Classes:")
for className in classNames:
    summaryText.append(f"- {className}")

summaryText.append("")
summaryText.append("Saved outputs:")
summaryText.append("- results/classification_report.txt")
summaryText.append("- results/confusion_matrix/confusion_matrix.png")
summaryText.append("- results/sample_predictions/sample_predictions.png")
summaryText.append("- results/gradcam/*.png")

with open("results/evaluation_summary.txt", "w") as f:
    f.write("\n".join(summaryText))

print("Saved evaluation summary to results/evaluation_summary.txt")